# Обучение головы регрессии рейтинга (SoFIFA)

Загружаем обученный трансформер из `mpp_mini_output/final_model`, подменяем голову на регрессионную (предсказание overall), **замораживаем энкодер** и обучаем только голову на рейтингах SoFIFA. Голова получает **эмбеддинги с выхода трансформера** (после attention): для каждого игрока один токен (player_id, position=0, team=0, form_stats=0) прогоняется через энкодер, выход подаётся в голову.

In [1]:
import sys
from pathlib import Path
import pickle

ROOT = Path(".").resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from safetensors.torch import load_file as load_safetensors

device = torch.device("mps" if torch.mps.is_available() else "cpu")
print("Device:", device)

Device: mps


In [2]:
# Пути
CHECKPOINT_DIR = ROOT / "notebooks" / "mpp_mini_output" / "final_model"
METADATA_DIR = ROOT / "notebooks" / "train_sample_processed" / "metadata"
SOFIFA_CSV = ROOT / "dataset" / "sofifa_players.csv"

In [3]:
# Словарь игроков/команд (как при обучении MPP)
def load_vocab_from_metadata(metadata_dir: Path) -> dict:
    with open(metadata_dir / "player_name2id.pickle", "rb") as f:
        player_name2id = pickle.load(f)
    with open(metadata_dir / "team_name2id.pickle", "rb") as f:
        team_name2id = pickle.load(f)
    n_players = len(player_name2id)
    n_teams = len(team_name2id)
    return {
        "player_name2id": player_name2id,
        "team_name2id": team_name2id,
        "player_pad_token_id": n_players + 1,
        "team_pad_token_id": n_teams,
    }

vocab = load_vocab_from_metadata(METADATA_DIR)
player_pad_token_id = vocab["player_pad_token_id"]
teams_vocab_size = vocab["team_pad_token_id"] + 1
print("player_pad_token_id:", player_pad_token_id, "teams_vocab_size:", teams_vocab_size)

player_pad_token_id: 6394 teams_vocab_size: 244


In [4]:
# SoFIFA: загрузка и сопоставление со словарём
from data.sofifa import load_sofifa_csv, build_rating_splits, SofifaRatingDataset

sofifa_df = load_sofifa_csv(SOFIFA_CSV)
train_df, eval_df = build_rating_splits(
    sofifa_df,
    vocab["player_name2id"],
    dev_ratio=0.15,
    seed=42,
    max_id=player_pad_token_id - 1,
)

train_ds = SofifaRatingDataset(train_df)
eval_ds = SofifaRatingDataset(eval_df)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
eval_loader = DataLoader(eval_ds, batch_size=64, shuffle=False)
print("Train samples:", len(train_ds), "Eval samples:", len(eval_ds))

Train samples: 1310 Eval samples: 231


In [5]:
# Модель: энкодер из чекпоинта + новая регрессионная голова
from models.pretrain import MaskedPlayerModel
from models.heads import RegressionHead

embed_size = 128
model = MaskedPlayerModel(
    embed_size=embed_size,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=39,
    players_vocab_size=player_pad_token_id,
    teams_vocab_size=teams_vocab_size,
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)

state = load_safetensors(CHECKPOINT_DIR / "model.safetensors")
encoder_state = {k.replace("encoder.", ""): v for k, v in state.items() if k.startswith("encoder.")}
model.encoder.load_state_dict(encoder_state, strict=True)

# Меняем голову на регрессию рейтинга; энкодер не обучаем
model.head = RegressionHead(embed_size=embed_size, output_dim=1, hidden_dim=128, pool="per_sequence")
for p in model.encoder.parameters():
    p.requires_grad = False

model = model.to(device)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable params (только голова):", n_trainable)

Trainable params (только голова): 16641


In [6]:
# Обучение только головы через RatingHeadTrainer
from training.rating_trainer import RatingHeadTrainer

trainer = RatingHeadTrainer(
    model,
    train_loader,
    eval_loader,
    output_dir=ROOT / "notebooks" / "mpp_mini_output",
    num_epochs=1000,
    lr=1e-3,
    weight_decay=0.01,
    device=device,
    logging_steps=10,
    save_best=True,
)
trainer.train()

Epoch 1/1000  train_loss=4473.0372  eval_rmse=55.8220
Epoch 10/1000  train_loss=259.8277  eval_rmse=15.3683
Epoch 20/1000  train_loss=170.0636  eval_rmse=12.6117
Epoch 30/1000  train_loss=129.7817  eval_rmse=10.8780
Epoch 40/1000  train_loss=93.9122  eval_rmse=10.0910
Epoch 50/1000  train_loss=78.8132  eval_rmse=9.4142
Epoch 60/1000  train_loss=71.2852  eval_rmse=9.1693
Epoch 70/1000  train_loss=61.5854  eval_rmse=9.0337
Epoch 80/1000  train_loss=54.0464  eval_rmse=8.8246
Epoch 90/1000  train_loss=49.6707  eval_rmse=8.7793
Epoch 100/1000  train_loss=51.0651  eval_rmse=8.5804
Epoch 110/1000  train_loss=44.6606  eval_rmse=8.4993
Epoch 120/1000  train_loss=43.3858  eval_rmse=8.4000
Epoch 130/1000  train_loss=45.8727  eval_rmse=8.4341
Epoch 140/1000  train_loss=42.5846  eval_rmse=8.4097
Epoch 150/1000  train_loss=38.8226  eval_rmse=8.1568
Epoch 160/1000  train_loss=37.6645  eval_rmse=8.2575
Epoch 170/1000  train_loss=37.9462  eval_rmse=8.1252
Epoch 180/1000  train_loss=37.0623  eval_rmse=8

7.318131666295488

In [11]:
# Пример предсказаний на eval (тот же forward, что при обучении: энкодер → голова)
model.eval()
form_stats_size = model.encoder.form_embeddings.in_features
with torch.no_grad():
    batch = next(iter(eval_loader))
    player_id = batch["player_id"].to(device)
    overall = batch["overall"].numpy()
    batch_size = player_id.size(0)
    input_ids = player_id.unsqueeze(1)
    position_ids = torch.zeros(batch_size, 1, dtype=torch.long, device=device)
    team_ids = torch.zeros(batch_size, 1, dtype=torch.long, device=device)
    form_stats = torch.zeros(batch_size, 1, form_stats_size, device=device)
    attention_mask = torch.ones(batch_size, 1, dtype=torch.long, device=device)
    enc_out, _ = model.encoder(input_ids, position_ids, team_ids, form_stats, attention_mask)
    pred = model.head(enc_out, attention_mask).squeeze(-1).cpu().numpy()

import pandas as pd
df = pd.DataFrame({"true": overall[:20], "pred": pred[:20].ravel()})
df["err"] = (df["true"] - df["pred"]).abs()
print(df.to_string())

    true       pred        err
0   85.0  70.825455  14.174545
1   75.0  74.033813   0.966187
2   75.0  69.714447   5.285553
3   70.0  74.876038   4.876038
4   70.0  73.588737   3.588737
5   71.0  75.567627   4.567627
6   67.0  68.544319   1.544319
7   69.0  77.961349   8.961349
8   70.0  72.929703   2.929703
9   79.0  74.833786   4.166214
10  70.0  77.777710   7.777710
11  65.0  80.263618  15.263618
12  75.0  76.959923   1.959923
13  82.0  70.410118  11.589882
14  70.0  77.506721   7.506721
15  67.0  75.702232   8.702232
16  70.0  70.037376   0.037376
17  71.0  73.499802   2.499802
18  61.0  71.217636  10.217636
19  81.0  71.454086   9.545914
